In [1]:
# Transformers example -> Sorting a sequence of numbers


The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.


In [2]:
import flax.nnx as nnx
import jax
import jax.numpy as jnp
import jax.random as jrandom
import optax

from probjax.nn import PosEncode, Transformer, RotaryPosEncode
from probjax.nn.layers.attention import flex_attention
from probjax.nn.pallas_kernels.attention_mask_bias import SeqLenMask

In [3]:
VOCAB_SIZE = 10

In [4]:
import jax.numpy as jnp
import jax.random as jrandom


def generate_data(key, n, T, vocab_size=10, lengths=None):
    if lengths is None:
        lengths = jnp.array(T) * jnp.ones(n, jnp.int32)

    m   = jnp.arange(T)[None, :] < lengths[:, None]                 # (n,T)
    tok = jrandom.randint(key, (n, T), 0, vocab_size, jnp.int32)  # (n,T)
    seq = jnp.where(m, tok, 9).astype(jnp.int32)               # (n,T)
    label = jnp.sort(seq, axis=-1)                             # (n,T)

    return seq[..., None], label[..., None]                    # (n,T,1),(n,T,1),(n,),(n,T)                # (n,T,1), (n,T,1), (n,T)

inputs, labels= generate_data(jax.random.PRNGKey(0), 1000, 10, VOCAB_SIZE)

In [5]:
inputs, labels= generate_data(jax.random.PRNGKey(0), 1000, 10, VOCAB_SIZE, lengths=jnp.array(2)*jnp.ones(1000, jnp.int32))

In [6]:
key = jrandom.PRNGKey(0)

In [17]:
class Model(nnx.Module):

    def __init__(self, dim,rngs, dropout_rate=0.):
        self.embed = nnx.Embed(VOCAB_SIZE, dim, rngs=rngs)
        self.pos_embed = RotaryPosEncode(dim,rngs=rngs)
        self.transformer = Transformer(dim, 8,8,8, widening_factor=4,rngs=rngs, dropout_rate=dropout_rate, attention_fn=flex_attention)
        self.output = nnx.Linear(dim, VOCAB_SIZE, rngs=rngs)

    def __call__(self, x, deterministic=False, mask=None):
        x = self.embed(x)
        x = jnp.squeeze(x,axis=-2)
        x = self.pos_embed(x)
        x = self.transformer(x,deterministic=deterministic, mask=mask)
        x = self.output(x)
        return x


In [18]:
model = Model(64, rngs=nnx.Rngs(0), dropout_rate=0.0)

In [19]:
params = nnx.state(model, nnx.Param)

In [20]:
optimizer = optax.radam(1e-4)
opt_state = optimizer.init(params)

In [1]:
max_train_seq = 128
batch_size = 2048
def loss_fn(params, key):
    nnx.update(model, params)
    key, key_sub, k_len = jax.random.split(key, 3)
    lengths = jax.random.binomial(k_len, max_train_seq, 0.3, shape=(batch_size,)).astype(jnp.int32) + 1
    #lengths = jax.random.choice(k_len, jnp.array([16,32,48,64]), (4096,), replace=True).astype(jnp.int32)
    inp_data, labels = generate_data(key_sub, batch_size, max_train_seq, lengths=lengths, vocab_size=VOCAB_SIZE)

    mask = SeqLenMask(lengths)#.dense(max_train_seq, max_train_seq, batch_size=4096)
    logits = model(inp_data, mask=mask)

    # xent per token
    labels_oh = jnp.squeeze(jax.nn.one_hot(labels, VOCAB_SIZE), -2)  # (B,S,V)
    token_loss = optax.softmax_cross_entropy(logits, labels_oh)      # (B,S)

    # mask out padded positions and average per *token*
    length_mask = jnp.arange(max_train_seq)[None, :] < lengths[:, None]          # (B,S)
    token_loss = jnp.where(length_mask, token_loss, 0.0)
    loss = token_loss.sum() / length_mask.sum()
    return loss

@jax.jit
def acc(params, inputs, outputs):
    nnx.update(model, params)
    inp_data, labels = inputs, outputs
    logits = model(inp_data)
    labels = jnp.squeeze(jax.nn.one_hot(labels, VOCAB_SIZE), -2)
    acc = (logits.argmax(axis=-1) == labels.argmax(-1)).mean()
    return acc

@jax.jit
def update(params, key, opt_state):
    loss, grads = jax.value_and_grad(loss_fn)(params, key)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return loss, params, opt_state

The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.


NameError: name 'jax' is not defined

In [22]:
key = jrandom.PRNGKey(0)

In [27]:

for i in range(10_000):
    key, subkey, key2 = jrandom.split(key, 3)
    loss, params, opt_state = update(params,key, opt_state)
    if (i % 1000) == 0:
        inputs, labels = generate_data(key, 1024,32, vocab_size=VOCAB_SIZE)
        accuracy = acc(params, inputs, labels)
        print(accuracy, loss)

0.7025757 0.0006234943
0.7222595 0.0004785222
0.732605 0.0036473332
0.7315674 0.00093291426
0.73309326 0.0006013654
0.74209595 0.00028685445
0.7487793 0.0003376447
0.7191162 0.001545763
0.72787476 0.00070548936
0.7303467 0.00050309993


In [28]:
model.eval()
nnx.update(model, params)

In [33]:
input = jax.random.randint(key+5, (1, 8,1),0, 10,dtype=jnp.int32)
outputs = model(input)
print(input[0,...,0])
print(outputs.argmax(-1)[0])
print(jnp.sort(input[0,...,0]))
print(jnp.all(outputs.argmax(-1)[0] == jnp.sort(input[0,...,0])))

[7 7 4 6 7 2 2 7]
[7 7 7 7 7 7 7 7]
[2 2 4 6 7 7 7 7]
False


In [22]:
jnp.allclose(outputs.argmax(-1)[0], jnp.sort(input[0,...,0]))

Array(False, dtype=bool)